# Tutorial: Plan 004.4 Grouped Duplicate Queries

**Audience:** Python users who know Archiver's catalog basics and want bounded duplicate inspection.

**Prerequisites:** an Archiver development environment and basic familiarity with `Catalog`.

**Learning goals:** construct immutable duplicate queries, bound groups and members independently, read complete summaries, understand deterministic ordering, and reuse a lazy query after a later scan.

**Outline:** setup → immutable bounds → bounded groups → complete summary → empty files → lazy evaluation → exercise → pitfalls and cleanup.

Every example uses disposable synthetic files. The API is observational: it identifies duplicate content but never selects a keeper or changes source files.

## 1. Create duplicate groups with visible ordering differences

The sample has four duplicate-content groups, one unique file, nested paths, and an empty-file group. The two five-byte groups tie on potential redundant bytes, so their complete content identities decide their order.

In [ ]:
from __future__ import annotations

import hashlib
from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog, DuplicateGroupQuery


temporary_directory = TemporaryDirectory(prefix="archiver-duplicate-query-")
workspace = Path(temporary_directory.name)
root = workspace / "source"
root.mkdir()


def write_group(directory: str, names: tuple[str, ...], content: bytes) -> None:
    group_directory = root / directory
    group_directory.mkdir(parents=True)
    for name in names:
        (group_directory / name).write_bytes(content)


write_group("large", ("z.bin", "a.bin", "m.bin"), b"large")
write_group("alpha", ("two.bin", "one.bin"), b"alpha")
write_group("bravo", ("two.bin", "one.bin"), b"bravo")
write_group("empty", ("two.bin", "one.bin"), b"")
(root / "unique.bin").write_bytes(b"unique")

catalog = Catalog.create(workspace / "catalog.sqlite")
scan_summary = catalog.scan_directory(root)
assert scan_summary.duplicate_content_group_count == 4
scan_summary

In [ ]:
[member.relative_path.as_posix() for member in catalog.files(root).fetch()]

## 2. Build immutable group and member bounds

`Catalog.duplicates(root)` creates a lazy `DuplicateGroupQuery`. New queries default to 20 groups and 20 displayed members per group. `limit()` and `member_limit()` return new values rather than changing the base query.

In [ ]:
base: DuplicateGroupQuery = catalog.duplicates(root)
preview = base.limit(2).member_limit(1)

bounds = {
    "base": (base.group_limit, base.member_row_limit),
    "preview": (preview.group_limit, preview.member_row_limit),
}
assert bounds == {"base": (20, 20), "preview": (2, 1)}
bounds

## 3. Fetch bounded groups with complete per-group metrics

Groups are ordered by potential redundant bytes descending, then full algorithm and digest ascending. Members use ascending complete relative paths. Although this preview displays one member, each view retains its complete instance count and redundant-byte metric.

In [ ]:
groups = preview.fetch()
rows = [
    {
        "digest": group.content_id.digest[:12],
        "size_bytes": group.size_bytes,
        "complete_instances": group.file_instance_count,
        "potential_redundant_bytes": group.potential_redundant_bytes,
        "displayed_paths": [member.relative_path.as_posix() for member in group.members],
    }
    for group in groups
]

tied_digests = sorted((hashlib.sha256(b"alpha").hexdigest(), hashlib.sha256(b"bravo").hexdigest()))
assert len(groups) == 2
assert rows[0]["displayed_paths"] == ["large/a.bin"]
assert rows[0]["complete_instances"] == 3
assert rows[0]["potential_redundant_bytes"] == 10
assert groups[1].content_id.digest == tied_digests[0]
rows

## 4. Ask for the complete summary

`summary()` ignores both display bounds. It describes all four duplicate groups and all nine duplicate file instances, even though the preview fetched only two groups with one member each.

In [ ]:
summary = preview.summary()
summary_values = {
    "duplicate_groups": summary.duplicate_content_group_count,
    "duplicate_file_instances": summary.duplicate_file_instance_count,
    "potential_redundant_bytes": summary.potential_redundant_bytes,
    "displayed_groups": len(groups),
    "displayed_members": sum(len(group.members) for group in groups),
}
assert summary_values == {
    "duplicate_groups": 4,
    "duplicate_file_instances": 9,
    "potential_redundant_bytes": 20,
    "displayed_groups": 2,
    "displayed_members": 2,
}
summary_values

In [ ]:
summary2 = base.summary()
groups2 = base.fetch()
summary_values = {
    "duplicate_groups": summary2.duplicate_content_group_count,
    "duplicate_file_instances": summary2.duplicate_file_instance_count,
    "potential_redundant_bytes": summary2.potential_redundant_bytes,
    "displayed_groups": len(groups2),
    "displayed_members": sum(len(group.members) for group in groups2),
}

summary_values

## 5. Empty content is still a duplicate identity

Two empty files share the SHA-256 identity of empty bytes. Their group remains visible even though removing an extra empty copy would represent zero potential redundant bytes.

In [ ]:
all_four = base.limit(4).member_limit(2).fetch()
empty_group = all_four[-1]
assert empty_group.content_id.digest == hashlib.sha256(b"").hexdigest()
assert empty_group.size_bytes == 0
assert empty_group.file_instance_count == 2
assert empty_group.potential_redundant_bytes == 0
[member.relative_path.as_posix() for member in empty_group.members]

## 6. Reuse a lazy query after a later successful scan

Construction validates the root but does not require it to have a catalog location or successful scan. Each terminal resolves current state when it runs.

In [ ]:
later_root = workspace / "later-source"
later_root.mkdir()
later_query = catalog.duplicates(later_root)

assert later_query.fetch() == ()
assert later_query.summary().duplicate_content_group_count == 0

(later_root / "first.txt").write_bytes(b"later duplicate")
(later_root / "nested").mkdir()
(later_root / "nested" / "second.txt").write_bytes(b"later duplicate")
catalog.scan_directory(later_root)

later_groups = later_query.fetch()
assert len(later_groups) == 1
[member.relative_path.as_posix() for member in later_groups[0].members]

## Exercise

Create a one-group, two-member view of the original root. Before running the solution, predict the displayed paths, complete instance count, and complete catalog summary.

In [ ]:
exercise_query = catalog.duplicates(root).limit(1).member_limit(2)
exercise_group = exercise_query.fetch()[0]
exercise_summary = exercise_query.summary()

answer = {
    "paths": [member.relative_path.as_posix() for member in exercise_group.members],
    "complete_instances_in_group": exercise_group.file_instance_count,
    "complete_group_count": exercise_summary.duplicate_content_group_count,
}
assert answer == {
    "paths": ["large/a.bin", "large/m.bin"],
    "complete_instances_in_group": 3,
    "complete_group_count": 4,
}
answer

In [ ]:
exercise_query.fetch()

## Pitfalls and extension

**Pitfall:** `len(query.fetch())` and `len(group.members)` are display counts, not complete totals. Use `query.summary()` and `group.file_instance_count`.

**Safety boundary:** there is deliberately no unbounded duplicate fetch, keeper selection, or deletion operation. Duplicate queries are read-only.

**Extension:** add another duplicate group with a different size, scan again, and predict where it appears under the fixed redundant-bytes ordering.

In [ ]:
catalog.close()
temporary_directory.cleanup()
print("Cleaned up the disposable catalogs and source directories.")